In [ ]:
# ─── Célula 1: Seleção do modelo ───────────────────────────────────────────────
# Execute esta célula PRIMEIRO e siga as recomendações antes de continuar.

MODELOS = {
    '1': {
        'id'            : 'pucpr/biobertpt-clin',
        'label'         : 'BioBERTpt-clin',
        'slug'          : 'BioBERTpt-clin',
        'secao'         : '2.3.2',
        'gpu_rec'       : 'T4 (Colab gratuito)',
        'batch_size'    : 16,
        'epochs'        : 5,
        'learning_rate' : 2e-5,
        'flash_attn'    : False,
        'aviso'         : None,
    },
    '2': {
        'id'            : 'pierreguillou/bert-base-cased-pt-lenerbr',
        'label'         : 'BERTimbau-leNER',
        'slug'          : 'BERTimbau-leNER',
        'secao'         : '2.3.3',
        'gpu_rec'       : 'T4 (Colab gratuito)',
        'batch_size'    : 16,
        'epochs'        : 5,
        'learning_rate' : 2e-5,
        'flash_attn'    : False,
        'aviso'         : None,
    },
    '3': {
        'id'            : 'jhu-clsp/mmBERT-base',
        'label'         : 'mmBERT',
        'slug'          : 'mmBERT-base',
        'secao'         : '2.3.4',
        'gpu_rec'       : 'A100 (Colab Pro)',
        'batch_size'    : 32,
        'epochs'        : 8,
        'learning_rate' : 1e-5,
        'flash_attn'    : False,
        'aviso'         : 'Requer GPU Ampere+ (A100 ou L4) — VRAM >16GB. Vá em Runtime → Change runtime type → A100 antes de continuar.',
    },
    '4': {
        'id'            : 'answerdotai/ModernBERT-base',
        'label'         : 'ModernBERT-base',
        'slug'          : 'ModernBERT-base',
        'secao'         : '2.3.5',
        'gpu_rec'       : 'A100 (Colab Pro)',
        'batch_size'    : 32,
        'epochs'        : 8,
        'learning_rate' : 1e-5,
        'flash_attn'    : True,
        'aviso'         : 'Requer GPU Ampere+ (A100 ou L4). Vá em Runtime → Change runtime type → A100 antes de continuar.',
    },
    '5': {
        'id'            : 'pierreguillou/ner-bert-large-cased-pt-lenerbr',
        'label'         : 'BERTimbau-leNER-large',
        'slug'          : 'BERTimbau-leNER-large',
        'secao'         : '2.3.6',
        'gpu_rec'       : 'L4 ou A100 (Colab Pro)',
        'batch_size'    : 8,
        'epochs'        : 5,
        'learning_rate' : 1e-5,
        'flash_attn'    : False,
        'aviso'         : 'Modelo large (335M params). Use L4 ou A100 — VRAM >16GB. Batch reduzido para 8.',
    },
}

print('=' * 60)
print('  AnonClin — Fine-tuning BERT para NER Clínico')
print('=' * 60)
print()
print('Escolha o modelo para treinar:')
print()
for k, m in MODELOS.items():
    print(f'  {k}) {m["label"]:45s} [{m["gpu_rec"]}]  — seção {m["secao"]}')
print()

escolha = input('Número (1-5): ').strip()
cfg = MODELOS[escolha]

MODEL_ID    = cfg['id']
MODEL_SLUG  = cfg['slug']
MODEL_LABEL = cfg['label']
BATCH_SIZE     = cfg['batch_size']
EPOCHS         = cfg['epochs']
LEARNING_RATE  = cfg['learning_rate']
FLASH_ATTN     = cfg['flash_attn']

print()
print('=' * 60)
print(f'  Modelo selecionado: {cfg["label"]} ({cfg["secao"]})')
print(f'  Model ID          : {MODEL_ID}')
print(f'  GPU recomendada   : {cfg["gpu_rec"]}')
print(f'  Batch size        : {BATCH_SIZE}')
print(f'  Epochs            : {EPOCHS}')
print(f'  Learning rate     : {LEARNING_RATE}')
print(f'  Flash Attention 2 : {"sim" if FLASH_ATTN else "não"}')
print('=' * 60)

if cfg['aviso']:
    print()
    print('⚠️  ATENÇÃO:', cfg['aviso'])
    print('   Troque o runtime ANTES de executar as próximas células.')
else:
    print()
    print('✅  Configuração OK para Colab gratuito (T4).')

In [ ]:
# ─── Célula 2: Detecção de ambiente e validação de GPU ─────────────────────────
import os, sys

try:
    import google.colab
    COLAB = True
    print('Ambiente: Google Colab')
except ImportError:
    COLAB = False
    print('Ambiente: Local')

import torch
GPU = torch.cuda.is_available()
print(f'GPU disponível: {GPU}')
if GPU:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap      = torch.cuda.get_device_properties(0).major  # compute capability major
    print(f'  {gpu_name} — VRAM: {vram_gb:.1f} GB — Compute: {cap}.x')

    if FLASH_ATTN and cap < 8:
        print()
        print('❌  ERRO: Este modelo requer GPU Ampere ou superior (compute 8.0+).')
        print(f'   GPU atual ({gpu_name}) é compute {cap}.x — Flash Attention 2 não suportado.')
        print('   Troque para A100 ou L4 em Runtime → Change runtime type.')
        raise SystemExit('GPU inadequada para o modelo selecionado.')
    elif FLASH_ATTN:
        print(f'  ✅  GPU compatível com Flash Attention 2.')
    else:
        print(f'  ✅  GPU OK para este modelo.')

In [ ]:
# ─── Célula 3: Instalação de dependências ──────────────────────────────────────
if COLAB:
    %pip install -q transformers datasets seqeval torch accelerate

    if FLASH_ATTN:
        print('Instalando Flash Attention 2 (pode levar 5-10 min)...')
        %pip install -q flash-attn --no-build-isolation
        print('Flash Attention 2 instalada.')

In [ ]:
# ─── Célula 4: Caminhos e repositório ──────────────────────────────────────────
if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE      = '/content/drive/MyDrive/Mestrado/Dissertação/AnonClin/Experimentos/Experimento 001'
    DRIVE_DATA_PATH = os.path.join(DRIVE_BASE, 'dados')
    OUTPUT_BASE     = '/content/modelos'  # local Colab — não ocupa cota do Drive

    TRAIN_PATH = os.path.join(DRIVE_DATA_PATH, 'train.conll')
    DEV_PATH   = os.path.join(DRIVE_DATA_PATH, 'dev.conll')
    TEST_PATH  = os.path.join(DRIVE_DATA_PATH, 'test.conll')

    REPO_URL = 'https://github.com/glauciohenriquesilva/anonimizacao-textos-clinicos-ptbr.git'
    REPO_DIR = '/content/anonclin'
    import subprocess, shutil

    if not os.path.exists(os.path.join(REPO_DIR, 'ner')):
        if os.path.exists(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f'Clone falhou:\n{result.stderr}')
        print('Clone OK')
    else:
        print('Repo já clonado')

    for pkg in [REPO_DIR, f'{REPO_DIR}/ner', f'{REPO_DIR}/ner/services']:
        init = os.path.join(pkg, '__init__.py')
        if not os.path.exists(init):
            open(init, 'w').close()

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)

    TRAIN_PATH  = os.path.join(REPO_ROOT, 'outputs', 'ner', 'train.conll')
    DEV_PATH    = os.path.join(REPO_ROOT, 'outputs', 'ner', 'dev.conll')
    TEST_PATH   = os.path.join(REPO_ROOT, 'outputs', 'ner', 'test.conll')
    OUTPUT_BASE = os.path.join(REPO_ROOT, 'outputs', 'ner', 'modelos')

OUTPUT_DIR = os.path.join(OUTPUT_BASE, MODEL_SLUG)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'train : {TRAIN_PATH} — existe: {os.path.exists(TRAIN_PATH)}')
print(f'dev   : {DEV_PATH}   — existe: {os.path.exists(DEV_PATH)}')
print(f'test  : {TEST_PATH}  — existe: {os.path.exists(TEST_PATH)}')
print(f'saída : {OUTPUT_DIR}')

In [ ]:
# ─── Célula 5: Treinamento ─────────────────────────────────────────────────────
from ner.services.bert import treinar_bert

# Garante OUTPUT_DIR correto mesmo se célula 4 não foi reexecutada após trocar de modelo
OUTPUT_DIR = os.path.join(OUTPUT_BASE, MODEL_SLUG)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Iniciando treino : {MODEL_LABEL}')
print(f'Model ID         : {MODEL_ID}')
print(f'Batch: {BATCH_SIZE} | Epochs: {EPOCHS} | Saída: {OUTPUT_DIR}')
print()

trainer, tokenizer, label2id, id2label = treinar_bert(
    model_id      = MODEL_ID,
    caminho_train = TRAIN_PATH,
    caminho_dev   = DEV_PATH,
    caminho_saida = OUTPUT_DIR,
    epochs        = EPOCHS,
    batch_size    = BATCH_SIZE,
    learning_rate = LEARNING_RATE,
)

print(f'\n✅ {MODEL_LABEL} salvo em: {OUTPUT_DIR}')

In [ ]:
# ─── Célula 6: Avaliação no conjunto de teste ──────────────────────────────────
# Usa word_ids() para alinhamento subtoken → palavra.
# Mais robusto que offsets de caractere — funciona com WordPiece (BERT) e BPE (mmBERT, ModernBERT).
from ner.services.bert import carregar_conll
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import classification_report, f1_score
import torch

tokenizer_eval = AutoTokenizer.from_pretrained(OUTPUT_DIR)
modelo_eval    = AutoModelForTokenClassification.from_pretrained(OUTPUT_DIR)
modelo_eval.eval()

device_str  = 'cuda' if torch.cuda.is_available() else 'cpu'
modelo_eval = modelo_eval.to(device_str)
id2label    = modelo_eval.config.id2label

teste  = carregar_conll(TEST_PATH)
y_true, y_pred = [], []

# Limite real de tokens lido do config do modelo (confiável para BERT e ModernBERT)
_model_max    = getattr(modelo_eval.config, 'max_position_embeddings', 512)
MAX_SUBTOKENS = _model_max - 2
print(f'MAX_SUBTOKENS: {MAX_SUBTOKENS} (max_position_embeddings={_model_max})')

for ex in teste:
    tokens_orig = ex['tokens']

    # Trunca se necessário (evita exceder limite do modelo)
    tokens_eval = tokens_orig
    while tokens_eval:
        n_sub = len(tokenizer_eval(tokens_eval, is_split_into_words=True,
                                   add_special_tokens=False)['input_ids'])
        if n_sub <= MAX_SUBTOKENS:
            break
        tokens_eval = tokens_eval[:-20]

    # Tokeniza e obtém word_ids — alinhamento direto subtoken → palavra
    enc      = tokenizer_eval(
        tokens_eval,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_SUBTOKENS + 2,
    )
    word_ids = enc.word_ids(batch_index=0)

    with torch.no_grad():
        logits = modelo_eval(**{k: v.to(device_str) for k, v in enc.items()}).logits[0]
    preds = torch.argmax(logits, dim=-1).cpu().tolist()

    # Primeiro subtoken de cada palavra define o label (padrão BIO)
    seen        = set()
    pred_labels = ['O'] * len(tokens_orig)
    for subtoken_idx, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen:
            continue
        seen.add(word_id)
        pred_labels[word_id] = id2label[preds[subtoken_idx]]

    y_true.append(ex['labels'])
    y_pred.append(pred_labels)

f1 = f1_score(y_true, y_pred)
print(f'Modelo  : {MODEL_LABEL} ({MODEL_ID})')
print(f'F1 Entity-level (micro): {f1:.4f}')
print()
print(classification_report(y_true, y_pred))

In [ ]:
# ─── Célula 7: Empacotar modelo para importar no AnonClin ─────────────────────
import shutil, glob

# Remove checkpoints intermediários — mantém só o modelo final (~700MB)
for ckpt in glob.glob(os.path.join(OUTPUT_DIR, 'checkpoint-*')):
    shutil.rmtree(ckpt)
print('Checkpoints removidos.')

zip_dir = '/content/modelos' if COLAB else OUTPUT_BASE
os.makedirs(zip_dir, exist_ok=True)
zip_path = os.path.join(zip_dir, f'bert_model_{MODEL_SLUG}')
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f'Arquivo gerado: {zip_path}.zip')

if COLAB:
    from google.colab import files
    files.download(f'{zip_path}.zip')
    print('Download iniciado automaticamente.')
else:
    print(f'Arquivo disponível em: {zip_path}.zip')
    print('Para importar no AnonClin: NER → Avaliação → upload do .zip')